<a href="https://colab.research.google.com/github/heavenmaker024/114-2PL-Repo61271012H/blob/main/HW4_PTT_GoogleSheet_RAG_%E6%B5%A901%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安裝必要套件 (精準鎖定版本以解決衝突，並使用最新的 google-genai)
!pip -q install pandas==2.2.2 requests==2.32.4 gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 google-genai gradio

import re
import time
import uuid
import random
from datetime import datetime
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

# 載入最新的 Google GenAI SDK
from google import genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe

# ==========================================
# 1. 系統設定與驗證
# ==========================================
print("🔄 正在驗證系統與金鑰...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

api_key = userdata.get("Geminiapikey")
if not api_key:
    raise ValueError("❌ 找不到 Colab Secret 金鑰。請先在左側密碼本新增 'Geminiapikey'。")

# 使用新版 SDK 初始化 Client
client = genai.Client(api_key=api_key)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1Hg-6uiw7Qr0HtqSzP8yBzv7dBH9uEE9gMgPNSAXRtR8/edit?gid=0#gid=0"
PTT_WORKSHEET_NAME = "ptt_car_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = ["post_id", "title", "url", "date", "author", "nrec", "created_at", "fetched_at", "content"]
PTT_CAR_INDEX = "https://www.ptt.cc/bbs/car/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")

# ==========================================
# 2. 試算表操作模組
# ==========================================
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update(values=[header], range_name="A1")
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update(values=[header], range_name="A1")
    elif values[0] != header:
        ws.clear()
        ws.update(values=[header], range_name="A1")
    return ws

def read_sheet_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns: df[col] = ""
    return df[header].fillna("")

def write_sheet_df(ws, df, header):
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns: df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("")
    for c in df_out.columns: df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)

ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)

# ==========================================
# 3. 強化防禦 PTT 爬蟲模組 (解決被踢下線的問題)
# ==========================================
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

# 建立具有重試機制的連線 Session
session = requests.Session()
# 當遇到 429(請求太多)、500、502、503、504 或連線中斷時，自動重試最多 5 次
retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update({"User-Agent": USER_AGENT})
session.cookies.update(PTT_COOKIES)

def get_soup(url):
    # 【關鍵】加入 0.5 ~ 2 秒的隨機延遲，偽裝成真人在點擊網頁
    time.sleep(random.uniform(0.5, 2.0))
    resp = session.get(url, timeout=20)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None

def parse_nrec(nrec_span):
    if not nrec_span: return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆": return 100
    if txt.startswith("X"):
        try: return -int(txt[1:])
        except: return -10
    try: return int(txt)
    except: return 0

def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a: continue
        title = a.get_text(strip=True)
        if "公告" in title: continue

        posts.append({
            "title": title,
            "url": urljoin("https://www.ptt.cc", a.get("href")),
            "author": item.select_one("div.author").get_text(strip=True) if item.select_one("div.author") else "",
            "date": item.select_one("div.date").get_text(strip=True) if item.select_one("div.date") else "",
            "nrec": parse_nrec(item.select_one("div.nrec span")),
        })
    return posts

def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main: return "", ""

    created_at = ""
    for m in main.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    for node in main.select("div.article-metaline, div.article-metaline-right, div.push, span.f2"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at

def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")

def crawl_and_update_ptt_car(pages=2):
    print(f"\n🕷️ 開始爬取 PTT 汽車版最新 {pages} 頁文章...")
    all_rows = []
    index_url = PTT_CAR_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}...")
        try:
            index_soup = get_soup(index_url)
            post_list = extract_post_list(index_soup)

            for p in post_list:
                try:
                    article_soup = get_soup(p["url"])
                    content, created_at = clean_ptt_content(article_soup)

                    if len(content) < 10: continue

                    all_rows.append({
                        "post_id": make_post_id(p["url"]), "title": p["title"], "url": p["url"],
                        "date": p["date"], "author": p["author"], "nrec": p["nrec"],
                        "created_at": created_at, "fetched_at": now_iso(), "content": content,
                    })
                except Exception as e:
                    print(f"⚠️ 跳過文章：{p.get('title', '')} (原因：{e})")

            prev_url = get_prev_index_url(index_soup)
            if not prev_url: break
            index_url = prev_url

        except Exception as e:
            print(f"❌ 讀取列表頁失敗: {e}")
            break

    new_df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次成功爬取 {len(new_df)} 篇有效文章。")

    print("🔄 正在與 Google Sheet 舊資料合併去重...")
    existing_df = read_sheet_df(ws_ptt, PTT_HEADER)
    combined_df = pd.concat([existing_df, new_df]).drop_duplicates(subset=["post_id"], keep="last")

    write_sheet_df(ws_ptt, combined_df, PTT_HEADER)
    print(f"✅ 寫入完成！目前資料庫共有 {len(combined_df)} 篇文章。")
    return combined_df

# 執行爬蟲更新 (預設爬 2 頁)
rag_source_df = crawl_and_update_ptt_car(pages=2)

# ==========================================
# 4. 建立 FAISS 向量索引
# ==========================================
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"\n📚 準備進入 RAG 系統的文章數：{len(rag_source_df)}")

print("⏳ 正在載入 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        text = f"標題：{title}\n作者：{str(row.get('author', ''))}\n日期：{str(row.get('date', ''))}\n內容：{str(row.get('content', ''))}"
        docs.append({"post_id": str(row.get("post_id", "")), "title": title, "url": str(row.get("url", "")), "text": text})
    return docs

def build_faiss_index(docs):
    if not docs: raise ValueError("沒有可建立索引的文件")
    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings

rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)
print(f"✅ RAG 索引建立完成！")

# ==========================================
# 5. Gemini 問答系統 (使用新版 google-genai)
# ==========================================
GEMINI_MODEL_NAME = "gemini-2.5-flash"

def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents: return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1: continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results

def query_rag(question, k=3, max_retries=3):
    docs = retrieve_docs(question, k=k)
    if not docs: return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join([f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs])

    prompt = f"""
你是一個根據 PTT 汽車版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    # 加入自動重試機制
    for attempt in range(max_retries):
        try:
            # 使用新版 SDK 呼叫 API
            response = client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=prompt
            )
            return response.text

        except Exception as e:
            error_msg = str(e)
            # 如果是 503 或 429 忙碌錯誤，則等待後重試
            if "503" in error_msg or "UNAVAILABLE" in error_msg or "429" in error_msg:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # 等待 1秒, 2秒...
                    print(f"⚠️ Google 伺服器忙碌中，等待 {wait_time} 秒後自動重試 (第 {attempt+1}/{max_retries} 次)...")
                    time.sleep(wait_time)
                else:
                    return f"❌ 抱歉，AI 伺服器目前大塞車，已自動重試 {max_retries} 次皆失敗，請稍後再試。"
            else:
                # 若是其他嚴重錯誤，直接回報
                return f"❌ 發生未知的 AI 錯誤：{error_msg}"

# ==========================================
# 6. 執行互動問答測試
# ==========================================
print("\n" + "="*50)
question = input("👉 請輸入問題 (例如：2026新款豐田有甚麼？)：")
print("\n🤖 AI 思考中...\n")
answer = query_rag(question, k=3)
print("=" * 50)
print(answer)
print("=" * 50)